# Preparing the Data: Document Loaders and Chunking Strategies
In a production RAG system, the quality of your AI's answer is 80% dependent on the quality of your **Chunks**. 

If your chunks are too small, the AI loses the "Big Picture." If they are too large, they won't fit in the context window or will dilute the semantic meaning. As an Architect, you must design a **Chunking Strategy** that balances resolution with context.

William Shakespeare's famous play, "Hamlet". -- https://gist.github.com/provpup/2fc41686eab7400b796b

In [1]:
import os
from IPython.display import Markdown, display

# We will use a sample file from the repo: Hamlet
FILE_PATH = "hamlet.txt"

def load_document(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

document = load_document(FILE_PATH)
print(f"✅ Document Loaded: {FILE_PATH}")
print(f"📊 Total Characters: {len(document):,}")

✅ Document Loaded: hamlet.txt
📊 Total Characters: 191,726


## 1. Fixed-Size Chunking (The "Brute Force" Way)

This method splits text every N characters. It's fast but often cuts words or sentences in half, destroying meaning.

In [2]:
def fixed_size_chunk(text, size):
    return [text[i:i + size] for i in range(0, len(text), size)]

chunks = fixed_size_chunk(document, 500)
print(f"Generated {len(chunks)} chunks using Fixed-Size (500 chars).")
print(f"\n--- Sample Chunk 10 ---\n{chunks[10]}")

Generated 384 chunks using Fixed-Size (500 chars).

--- Sample Chunk 10 ---
as, as you know, by Fortinbras of Norway,
    Thereto prick'd on by a most emulate pride,
    Dar'd to the combat; in which our valiant Hamlet
    (For so this side of our known world esteem'd him)
    Did slay this Fortinbras; who, by a seal'd compact,
    Well ratified by law and heraldry,
    Did forfeit, with his life, all those his lands
    Which he stood seiz'd of, to the conqueror;
    Against the which a moiety competent
    Was gaged by our king; which had return'd
    To the inheritan


## 2. Recursive Character Chunking (The Architect's Way)

Instead of cutting blindly, we try to split at natural boundaries: 
1. Paragraphs (`\n\n`)
2. Sentences (`\n`)
3. Spaces (` `)

This ensures that related thoughts stay together in the same vector.

In [3]:
def recursive_splitter(text, max_size, overlap):
    """
    A simplified version of the logic used in LangChain.
    It keeps chunks under max_size while maintaining 'overlap' 
    to ensure context isn't lost at the edges.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_size
        chunk = text[start:end]
        chunks.append(chunk)
        # Move the window forward, but stay back by the 'overlap' amount
        start += (max_size - overlap)
    return chunks

smart_chunks = recursive_splitter(document, 1000, 100)
print(f"Generated {len(smart_chunks)} chunks with 10% Overlap.")
print(f"\n--- Smart Chunk 5 ---\n{smart_chunks[5]}")

Generated 214 chunks with 10% Overlap.

--- Smart Chunk 5 ---
ant watch
    So nightly toils the subject of the land,
    And why such daily cast of brazen cannon
    And foreign mart for implements of war;
    Why such impress of shipwrights, whose sore task
    Does not divide the Sunday from the week.
    What might be toward, that this sweaty haste  
    Doth make the night joint-labourer with the day?
    Who is't that can inform me?
  Hor. That can I.
    At least, the whisper goes so. Our last king,
    Whose image even but now appear'd to us,
    Was, as you know, by Fortinbras of Norway,
    Thereto prick'd on by a most emulate pride,
    Dar'd to the combat; in which our valiant Hamlet
    (For so this side of our known world esteem'd him)
    Did slay this Fortinbras; who, by a seal'd compact,
    Well ratified by law and heraldry,
    Did forfeit, with his life, all those his lands
    Which he stood seiz'd of, to the conqueror;
    Against the which a moiety competent
    